# 저장 모델 ML + DL 확률 결합

`fusion.yaml`에서 학습된 ML·DL artifact 경로, 결합 방식, 가중치, threshold를 지정합니다. 기본 동작은 저장 모델을 불러와 추론만 하며 기반 모델을 재학습하지 않습니다. 모델 경로를 실제 run으로 바꾼 뒤 실행하세요.

In [ ]:
import json
from pathlib import Path

import pandas as pd

repo = Path.cwd().resolve()
while not (repo / "Members/sang/model/fusion.yaml").is_file():
    if repo.parent == repo:
        raise RuntimeError("저장소 루트에서 노트북을 실행하세요")
    repo = repo.parent
fusion_config = repo / "Members/sang/model/fusion.yaml"
from Members.sang.model.fusion import run_fusion  # noqa: E402

## Validation 또는 test 추론

수동 가중치는 합이 1이어야 합니다. `mode: learn`이면 저장 모델을 고정하고 validation 환자 확률만으로 전역 가중치를 학습합니다. 그 뒤 test를 지정하면 같은 weights를 적용합니다. 가중치를 학습한 validation 점수는 독립적인 최종 추정이 아닙니다.

In [ ]:
fusion_run = run_fusion(fusion_config, split_name="validation")
print(fusion_run)
print(json.loads((fusion_run / "metrics.json").read_text()))
display(pd.read_csv(fusion_run / "validation_predictions.csv").head())

## 재사용

생성된 `fusion_weights.json`과 `resolved_config.yaml`에 선택된 모델 run ID, 가중치, threshold, 입력 설정이 저장됩니다. 같은 결합을 다시 추론하려면 `fusion.yaml`의 `mode: saved`와 `saved_run_dir`를 설정해 validation 가중치 학습을 반복하지 않습니다. 최종 test 평가는 `evaluation/eval.yaml`의 split을 `test`로 설정해 별도로 수행하세요.